# Bài 1 – Phân loại táo, chuối và cam

Notebook thực hiện toàn bộ quy trình: kiểm tra dữ liệu Fruits-360, minh họa ảnh, trích đặc trưng màu, huấn luyện KNN/SVM và hiển thị kết quả. Mã xử lý chính nằm trong `tuan6/bai1_phan_loai_trai_cay.py`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import cv2
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'tuan6').exists() and (ROOT.parent / 'tuan6').exists():
    ROOT = ROOT.parent
assert (ROOT / 'tuan6').exists(), 'Hãy chạy notebook từ thư mục gốc của repository.'
DATA = ROOT / 'fruits-360-100x100-main' / 'Training'
OUTPUT = ROOT / 'tuan6' / 'output_bai1'
ROOT

## 1. Kiểm tra dữ liệu

In [ ]:
classes = {
    'Táo': 'Apple Braeburn 1',
    'Chuối': 'Banana 1',
    'Cam': 'Orange 1',
}
counts = {name: len(list((DATA / folder).glob('*.jpg'))) for name, folder in classes.items()}
pd.Series(counts, name='Số ảnh').to_frame()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (name, folder) in zip(axes, classes.items()):
    path = next((DATA / folder).glob('*.jpg'))
    image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(image)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()

## 2. Huấn luyện và đánh giá

Chương trình resize ảnh về 128×128, trích Mean/Std RGB, Mean/Std HSV, Histogram HSV và Entropy; sau đó chia Train/Test 80/20 có phân tầng và đánh giá bốn thí nghiệm.

In [ ]:
command = [
    sys.executable, str(ROOT / 'tuan6' / 'bai1_phan_loai_trai_cay.py'),
    '--data', str(DATA), '--output', str(OUTPUT), '--seed', '42'
]
subprocess.run(command, check=True)

## 3. Bảng kết quả

In [ ]:
features = pd.read_csv(OUTPUT / '02_bang_dac_trung.csv')
metrics = pd.read_csv(OUTPUT / '03_bang_so_sanh_mo_hinh.csv')
print('Kích thước bảng đặc trưng:', features.shape)
display(metrics)

In [ ]:
result_images = [
    ('Tiền xử lý', '01_minh_hoa_tien_xu_ly.png'),
    ('So sánh mô hình', '04_bieu_do_so_sanh.png'),
    ('Ma trận nhầm lẫn', '05_confusion_matrix.png'),
    ('Dự đoán tập kiểm thử', '06_anh_du_doan_tap_test.png'),
]
for title, filename in result_images:
    image = plt.imread(OUTPUT / filename)
    plt.figure(figsize=(14, 7))
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

## 4. Nhận xét

Bốn cấu hình đều đạt Accuracy và F1-macro bằng 1,00 trên tập kiểm thử 293 ảnh. Với ba lớp trái cây có màu sắc đặc trưng và nền ảnh đồng nhất, các đặc trưng màu đã đủ khả năng tách lớp. Khi áp dụng cho ảnh thực tế có nền phức tạp hoặc ánh sáng thay đổi, cần bổ sung tăng cường dữ liệu, tách nền và kiểm thử trên tập độc lập.